In [2]:
# 数据读取与初检
from pathlib import Path
import pandas as pd

# 推断数据路径（在scripts目录下相对../raw）
data_path_candidates = [
    Path("../raw/gkx_20201231.csv"),
    Path("../../raw/gkx_20201231.csv"),
    Path("Assignment1/raw/gkx_20201231.csv"),
]

for candidate in data_path_candidates:
    if candidate.exists():
        data_path = candidate
        break
else:
    raise FileNotFoundError("未找到 gkx_20201231.csv，请检查路径")

# 读取数据
raw_df = pd.read_csv(data_path, low_memory=False)

# 日期转换（若存在DATE列）
if "DATE" in raw_df.columns:
    raw_df["DATE"] = pd.to_datetime(raw_df["DATE"], errors="coerce")
    raw_df = raw_df.sort_values("DATE")

# 基础信息
n_rows, n_cols = raw_df.shape
print(f"数据维度: {n_rows} 行, {n_cols} 列")
if "DATE" in raw_df.columns:
    print(
        "日期范围:",
        raw_df["DATE"].min(),
        "→",
        raw_df["DATE"].max(),
    )

print("\n列名预览:", list(raw_df.columns[:50]), "...")

# 缺失率Top 10
missing_rate = raw_df.isna().mean().sort_values(ascending=False)
print("\n缺失率Top10:\n", missing_rate.head(10))

# 数值列基本统计（前5列示例）
numeric_cols = raw_df.select_dtypes(include="number").columns
if len(numeric_cols) > 0:
    display(raw_df[numeric_cols[:5]].describe(percentiles=[0.01, 0.5, 0.99]))

# 前几行数据预览
display(raw_df.head())


数据维度: 4345508 行, 101 列
日期范围: 1970-01-01 00:00:00.019260130 → 1970-01-01 00:00:00.020201231

列名预览: ['permno', 'DATE', 'mvel1', 'RET', 'prc', 'SHROUT', 'beta', 'betasq', 'chmom', 'dolvol', 'idiovol', 'indmom', 'mom1m', 'mom6m', 'mom12m', 'mom36m', 'mve0', 'pricedelay', 'turn', 'absacc', 'acc', 'age', 'agr', 'cashdebt', 'cashpr', 'cfp', 'cfp_ia', 'chatoia', 'chcsho', 'chempia', 'chinv', 'chpmia', 'convind', 'currat', 'depr', 'divi', 'divo', 'dy', 'egr', 'ep', 'gma', 'grcapx', 'grltnoa', 'herf', 'hire', 'invest', 'lev', 'lgr', 'mve_ia', 'operprof'] ...

缺失率Top10:
 realestate    0.756666
rd_sale       0.673821
rd_mve        0.668240
secured       0.637079
stdcf         0.636829
stdacc        0.636829
roavol        0.531064
orgcap        0.521404
grltnoa       0.520301
pchsaleinv    0.509615
dtype: float64


,permno,mvel1,RET,prc,SHROUT
count,4.345508e+06,4.341591e+06,4.345508e+06,4.324990e+06,4.345508e+06
mean,5.325557e+04,1.247499e+06,1.083914e-02,3.114100e+01,4.502779e+04
std,2.874066e+04,6.074550e+06,1.740540e-01,1.347706e+03,2.316850e+05
min,1.000000e+04,0.000000e+00,-1.988095e+00,7.800000e-03,0.000000e+00
1%,1.025300e+04,9.494938e+02,-3.863640e-01,2.656250e-01,2.000000e+02
50%,5.669600e+04,8.348208e+04,0.000000e+00,1.370000e+01,8.240000e+03
99%,9.268100e+04,2.472807e+07,5.521721e-01,1.220000e+02,6.121904e+05
max,9.343600e+04,1.735365e+08,2.400000e+01,3.478150e+05,2.920640e+07


,permno,DATE,mvel1,RET,prc,SHROUT,beta,betasq,chmom,dolvol,...,baspread,ill,maxret,retvol,std_dolvol,std_turn,zerotrade,sic2,bm,bm_ia
0,10006,1970-01-01 00:00:00.019260130,65400.000,0.032732,110.25,600,NaN,NaN,NaN,NaN,...,0.006857,NaN,NaN,NaN,NaN,NaN,0.000066,NaN,NaN,NaN
338,13960,1970-01-01 00:00:00.019260130,1629.375,0.272727,1.75,1185,NaN,NaN,NaN,NaN,...,0.181818,NaN,NaN,NaN,NaN,NaN,0.000058,NaN,NaN,NaN
337,13952,1970-01-01 00:00:00.019260130,11638.375,0.081272,38.25,329,NaN,NaN,NaN,NaN,...,0.031634,NaN,NaN,NaN,NaN,NaN,0.000006,NaN,NaN,NaN
336,13944,1970-01-01 00:00:00.019260130,6256.250,0.202797,43.00,175,NaN,NaN,NaN,NaN,...,0.041958,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
335,13936,1970-01-01 00:00:00.019260130,5015.000,-0.059322,55.50,85,NaN,NaN,NaN,NaN,...,0.000000,NaN,NaN,NaN,NaN,NaN,0.000019,NaN,NaN,NaN


## 基于初检结果的下一步计划
上面的输出显示：
- 数据量大（约434万行、101列），按`DATE`排序后日期字段格式化为时间戳。
- 缺失较严重的特征（如realestate、rd_sale等）缺失率>50%，需要缺失值策略（删除列或填补）。
- 价格/市值等列存在长尾与极端值，后续正则化模型需标准化。

因此，下一步先做“初步清洗与特征准备”：删除指定无效列、明确目标`RET`、抽取特征列、查看特征缺失率和行级缺失分布，为后续缺失处理与标准化做决策。

In [3]:
# 初步清洗与特征准备
import numpy as np

cols_del = ['SHROUT', 'mve0', 'prc', 'permno', 'DATE', 'sic2']
target_col = 'RET'

if target_col not in raw_df.columns:
    raise KeyError('RET 不在数据列中，请检查数据字典')

# 复制并按日期排序，避免未来信息泄漏
work_df = raw_df.copy()
if 'DATE' in work_df.columns:
    work_df = work_df.sort_values('DATE')

# 目标与特征列划分
drop_cols = [c for c in cols_del if c in work_df.columns]
feature_cols = [c for c in work_df.columns if c not in drop_cols + [target_col]]
features = work_df[feature_cols]
y = work_df[target_col]

print(f"保留特征列数: {len(feature_cols)}（删除列: {drop_cols}；目标列: {target_col}）")

# 特征缺失率（Top10）
feature_missing = features.isna().mean().sort_values(ascending=False)
print("\n特征缺失率Top10:")
print(feature_missing.head(10))

# 行级缺失占比分布（抽样以节省计算）
sample_n = min(len(features), 50000)
row_missing_frac = features.sample(sample_n, random_state=42).isna().mean(axis=1)
print("\n行级缺失占比分布（抽样" + str(sample_n) + "行）：")
print(row_missing_frac.describe(percentiles=[0.5, 0.9, 0.99])) 

# 示例行预览
display(features.head())
display(y.head())


保留特征列数: 94（删除列: ['SHROUT', 'mve0', 'prc', 'permno', 'DATE', 'sic2']；目标列: RET）

特征缺失率Top10:
realestate    0.756666
rd_sale       0.673821
rd_mve        0.668240
secured       0.637079
stdcf         0.636829
stdacc        0.636829
roavol        0.531064
orgcap        0.521404
grltnoa       0.520301
pchsaleinv    0.509615
dtype: float64

行级缺失占比分布（抽样50000行）：
count    50000.000000
mean         0.333408
std          0.335975
min          0.000000
50%          0.170213
90%          0.797872
99%          0.925532
max          0.978723
dtype: float64


,mvel1,beta,betasq,chmom,dolvol,idiovol,indmom,mom1m,mom6m,mom12m,...,ms,baspread,ill,maxret,retvol,std_dolvol,std_turn,zerotrade,bm,bm_ia
0,65400.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.006857,NaN,NaN,NaN,NaN,NaN,0.000066,NaN,NaN
27,21621.25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.040000,NaN,NaN,NaN,NaN,NaN,0.000001,NaN,NaN
247,132358.25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.012978,NaN,NaN,NaN,NaN,NaN,0.000028,NaN,NaN
1,11200.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.018018,NaN,NaN,NaN,NaN,NaN,0.000003,NaN,NaN
2,23400.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.004158,NaN,NaN,NaN,NaN,NaN,0.000023,NaN,NaN


0      0.032732
27     0.000000
247    0.065903
1      0.017857
2      0.161667
Name: RET, dtype: float64

## 缺失值处理策略与下一步
上面结果显示：
- 个别特征缺失率 >60%（realestate、rd_sale、rd_mve 等），直接保留会降低有效样本维度。
- 行级缺失分布中位数约17%，90分位约80%，存在高缺失行；直接全量行删除会损失太多，但应剔除缺失极端行。

因此，先进行：
1) 列删：剔除缺失率 >0.6 的特征。
2) 行筛：剔除行缺失占比 >0.9 的极端行。
3) 数值填补：对剩余特征用中位数填补（后续可改为按期截面中位数以降低偏移）。
4) 保留列名并输出新形状，为后续标准化和时间序列CV做准备。


In [4]:
# 缺失值初步处理（列删+行筛+中位数填补）
from sklearn.impute import SimpleImputer

# 1) 列删：缺失率>0.6 的特征
high_missing_cols = feature_missing[feature_missing > 0.6].index.tolist()
features_step1 = features.drop(columns=high_missing_cols)
print(f"删除高缺失列 {len(high_missing_cols)} 个，剩余特征列 {features_step1.shape[1]} 个")

# 2) 行筛：缺失占比>0.9 的行剔除
row_missing_frac_step1 = features_step1.isna().mean(axis=1)
row_keep_mask = row_missing_frac_step1 <= 0.9
features_step2 = features_step1.loc[row_keep_mask]
y_step2 = y.loc[row_keep_mask]
print(f"剔除缺失>90%行 {(~row_keep_mask).sum()} 条，剩余 {features_step2.shape[0]} 行")

# 3) 中位数填补（数值列）
imputer = SimpleImputer(strategy='median')
X_imputed_array = imputer.fit_transform(features_step2)
X_imputed = pd.DataFrame(X_imputed_array, columns=features_step2.columns, index=features_step2.index)
print("填补完成：", X_imputed.shape)

# 预览
print("\n填补后特征示例:")
display(X_imputed.head())
print("\n对应目标示例:")
display(y_step2.head())


删除高缺失列 6 个，剩余特征列 88 个
剔除缺失>90%行 54195 条，剩余 4291313 行
填补完成： (4291313, 88)

填补后特征示例:


,mvel1,beta,betasq,chmom,dolvol,idiovol,indmom,mom1m,mom6m,mom12m,...,ms,baspread,ill,maxret,retvol,std_dolvol,std_turn,zerotrade,bm,bm_ia
504,5312.50,0.948461,0.903086,-0.004969,10.582168,0.049358,0.102745,0.011905,0.023256,0.053845,...,4.0,0.025226,2.906579e-06,0.073171,0.031768,1.058389,4.779582,1.957494e-08,0.64559,-0.105447
503,22110.00,0.948461,0.903086,-0.004969,10.582168,0.049358,0.102745,0.002278,0.023256,0.053845,...,4.0,0.009646,1.011980e-07,0.031142,0.009619,1.133733,6.069652,1.347701e-08,0.64559,-0.105447
508,27487.50,0.948461,0.903086,-0.004969,10.582168,0.049358,0.102745,0.001344,0.023256,0.053845,...,4.0,0.012694,1.442342e-07,0.019074,0.010184,1.223397,5.798691,8.400000e-01,0.64559,-0.105447
502,9834.00,0.948461,0.903086,-0.004969,7.280077,0.049358,0.102745,-0.083333,0.023256,0.053845,...,4.0,0.015453,3.919852e-06,0.062500,0.023225,0.770209,0.453796,8.801688e-08,0.64559,-0.105447
501,2535.75,0.948461,0.903086,-0.004969,2.386467,0.049358,0.102745,0.050000,0.023256,0.053845,...,4.0,0.037705,5.361814e-06,0.125000,0.034121,1.366994,9.854540,1.092000e+01,0.64559,-0.105447



对应目标示例:


504    0.129412
503    0.031818
508   -0.066849
502   -0.045455
501   -0.102041
Name: RET, dtype: float64

## 结果小结与下一步
- 已删除高缺失特征 6 个，保留 88 个特征；删除缺失>90% 的行后仍有约 429 万行数据。
- 特征已中位数填补，当前 `X_imputed`、`y_step2` 无缺失，可直接进入标准化与时间序列CV。
- 为避免泄漏，后续标准化应在每个训练折内 `fit`，在验证/测试折仅 `transform`。
- 时间序列切分需用 `DATE`（以及可选 `permno` 作为ID）保留下来；收益可做双侧截尾减轻极端值影响。


In [5]:
# 保存分割索引并对收益截尾

def winsorize_series(s, lower=0.005, upper=0.995):
    lo, hi = s.quantile([lower, upper])
    return s.clip(lo, hi)

# DATE / permno 用于时间序列切分与标识
dates_for_split = work_df.loc[X_imputed.index, 'DATE'] if 'DATE' in work_df.columns else None
permno_for_id = work_df.loc[X_imputed.index, 'permno'] if 'permno' in work_df.columns else None

# 对收益做双侧截尾，减轻极端值影响
y_wins = winsorize_series(y_step2, lower=0.005, upper=0.995)

print("分割索引可用:", dates_for_split is not None)
if permno_for_id is not None:
    print("permno 已保留，可用于ID对齐")
print("截尾后目标示例:")
display(y_wins.head())


分割索引可用: True
permno 已保留，可用于ID对齐
截尾后目标示例:


504    0.129412
503    0.031818
508   -0.066849
502   -0.045455
501   -0.102041
Name: RET, dtype: float64

## 时间序列递归CV与基线模型（示例）
- 使用 `DATE` 做时间有序切分，避免前视偏差；示例采用 3 折扩展窗口（可调）。
- 先对样本下采样以便快速试跑（可关闭采样跑全量，但耗时大）。
- 每折：训练内 `StandardScaler` 拟合，验证集仅 transform；训练 OLS、Ridge、LASSO，输出 MSE、R²。
- 视需要调整折数、窗口比例、正则化超参网格。

In [ ]:
# 时间序列递归CV + 基线模型试跑
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.metrics import mean_squared_error, r2_score

# 可调参数
use_sample = True
max_rows_for_demo = 200000  # 为避免长时间运行，先抽样；如需全量可设为False或调大
n_folds = 3
train_min_frac = 0.6  # 前60%日期作为最小训练起点
alpha_grid = [0.1, 1.0, 10.0, 100.0]

# 1) 抽样（可关闭）
if use_sample and len(X_imputed) > max_rows_for_demo:
    sample_idx = X_imputed.sample(max_rows_for_demo, random_state=42).index
    X_cv = X_imputed.loc[sample_idx]
    y_cv = y_wins.loc[sample_idx]
    dates_cv = dates_for_split.loc[sample_idx]
else:
    X_cv = X_imputed
    y_cv = y_wins
    dates_cv = dates_for_split

# 2) 按日期排序
order = np.argsort(dates_cv.values)
X_cv = X_cv.iloc[order]
y_cv = y_cv.iloc[order]
dates_cv = dates_cv.iloc[order]

# 3) 构造时间折
unique_dates = np.sort(dates_cv.unique())
folds = []
for i in range(n_folds):
    val_start = int(len(unique_dates) * (train_min_frac + i * (1 - train_min_frac) / n_folds))
    val_end = int(len(unique_dates) * (train_min_frac + (i + 1) * (1 - train_min_frac) / n_folds))
    val_end = min(val_end, len(unique_dates))
    train_cut = unique_dates[val_start]
    val_cut = unique_dates[val_end - 1] if val_end > val_start else unique_dates[-1]
    train_mask = dates_cv <= train_cut
    val_mask = (dates_cv > train_cut) & (dates_cv <= val_cut)
    if val_mask.sum() == 0:
        continue
    folds.append((train_mask, val_mask))

print(f"折数: {len(folds)}，示例每折验证区间基于日期量分位切分")

# 4) 逐折训练与评估
results = []
for fold_id, (train_mask, val_mask) in enumerate(folds, 1):
    X_train, X_val = X_cv.loc[train_mask], X_cv.loc[val_mask]
    y_train, y_val = y_cv.loc[train_mask], y_cv.loc[val_mask]

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_val_s = scaler.transform(X_val)

    models = {
        "OLS": LinearRegression(),
        "Ridge": RidgeCV(alphas=alpha_grid, store_cv_values=False),
        "LASSO": LassoCV(alphas=alpha_grid, max_iter=2000)
    }

    for name, model in models.items():
        model.fit(X_train_s, y_train)
        pred = model.predict(X_val_s)
        mse = mean_squared_error(y_val, pred)
        r2 = r2_score(y_val, pred)
        results.append({"fold": fold_id, "model": name, "mse": mse, "r2": r2, "n_train": len(y_train), "n_val": len(y_val)})
        print(f"Fold {fold_id} {name}: MSE={mse:.6f}, R2={r2:.4f}, n_train={len(y_train)}, n_val={len(y_val)}")

results_df = pd.DataFrame(results)
display(results_df)
